# Create platemap layout figures for visualization

NOTE: This notebook utilizes the R environment as installed from the main README.

## Import libraries

In [1]:
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(platetools))

Warning message:
“package ‘ggplot2’ was built under R version 4.2.3”


Warning message:
“package ‘platetools’ was built under R version 4.2.3”


## Set paths and variables

In [2]:
# Define the path to the platemaps folder
platemaps_folder <- "metadata/platemaps"

# Set up the file naming pattern for the platemap files (CSVs)
platemap_files <- list.files(path = platemaps_folder, pattern = "_platemap\\.csv$", full.names = TRUE)
print(platemap_files)

# Set output directory for the layout figures
output_fig_dir <- file.path("platemap_figures")

# Create the directory if it doesn't exist, ignore if it already exists
dir.create(output_fig_dir, showWarnings = FALSE, recursive = TRUE)

# Set suffix for output figures
platemap_suffix <- "_platemap_figure.png"

# Instantiate empty list for the plate layout names + output paths 
output_platemap_files <- list()

# Extract plate name and add output path for figure
for (platemap_file in platemap_files) {
    # Extract the base file name
    plate <- basename(platemap_file)
    
    # Remove the '_platemap.csv' suffix
    plate <- stringr::str_remove(plate, "_platemap\\.csv")
    
    # (Optional) Ensure there is no .csv extension remaining (shouldn't be needed after the above line)
    plate <- stringr::str_remove(plate, "\\.csv$")
    
    # Add to output_platemap_files with the cleaned plate name and output path
    output_platemap_files[[plate]] <- file.path(output_fig_dir, paste0(plate, platemap_suffix))
}

print(output_platemap_files)

 [1] "metadata/platemaps/Assay_Plate10_platemap.csv"
 [2] "metadata/platemaps/Assay_Plate11_platemap.csv"
 [3] "metadata/platemaps/Assay_Plate12_platemap.csv"
 [4] "metadata/platemaps/Assay_Plate13_platemap.csv"
 [5] "metadata/platemaps/Assay_Plate1_platemap.csv" 
 [6] "metadata/platemaps/Assay_Plate2_platemap.csv" 
 [7] "metadata/platemaps/Assay_Plate3_platemap.csv" 
 [8] "metadata/platemaps/Assay_Plate4_platemap.csv" 
 [9] "metadata/platemaps/Assay_Plate5_platemap.csv" 
[10] "metadata/platemaps/Assay_Plate6_platemap.csv" 
[11] "metadata/platemaps/Assay_Plate7_platemap.csv" 
[12] "metadata/platemaps/Assay_Plate8_platemap.csv" 
[13] "metadata/platemaps/Assay_Plate9_platemap.csv" 


$Assay_Plate10
[1] "platemap_figures/Assay_Plate10_platemap_figure.png"

$Assay_Plate11
[1] "platemap_figures/Assay_Plate11_platemap_figure.png"

$Assay_Plate12
[1] "platemap_figures/Assay_Plate12_platemap_figure.png"

$Assay_Plate13
[1] "platemap_figures/Assay_Plate13_platemap_figure.png"

$Assay_Plate1
[1] "platemap_figures/Assay_Plate1_platemap_figure.png"

$Assay_Plate2
[1] "platemap_figures/Assay_Plate2_platemap_figure.png"

$Assay_Plate3
[1] "platemap_figures/Assay_Plate3_platemap_figure.png"

$Assay_Plate4
[1] "platemap_figures/Assay_Plate4_platemap_figure.png"

$Assay_Plate5
[1] "platemap_figures/Assay_Plate5_platemap_figure.png"

$Assay_Plate6
[1] "platemap_figures/Assay_Plate6_platemap_figure.png"

$Assay_Plate7
[1] "platemap_figures/Assay_Plate7_platemap_figure.png"

$Assay_Plate8
[1] "platemap_figures/Assay_Plate8_platemap_figure.png"

$Assay_Plate9
[1] "platemap_figures/Assay_Plate9_platemap_figure.png"



## Load in the platemap layout information

In [3]:
# Load in all platemap CSV files
platemap_dfs <- list()
for (plate in names(output_platemap_files)) {
    # Find the umap file associated with the plate
    platemap_file <- platemap_files[basename(platemap_files) == paste0(plate, "_platemap.csv")]
    
    # Load in the umap data
    df <- readr::read_csv(
    platemap_file,
    col_types = readr::cols(.default = "c")
)

    platemap_dfs[[plate]] <- df 
}

print(platemap_dfs)

$Assay_Plate10
# A tibble: 222 × 6
   cell_line row   column well  seeding_density condition    
   <chr>     <chr> <chr>  <chr> <chr>           <chr>        
 1 SKNAS-EV  C     3      C03   1000            synthemax_PFA
 2 SKNAS-EV  C     4      C04   1000            synthemax_PFA
 3 SKNAS-EV  C     5      C05   2000            synthemax_PFA
 4 SKNAS-EV  C     6      C06   2000            synthemax_PFA
 5 SKNAS-EV  C     7      C07   4000            synthemax_PFA
 6 SKNAS-EV  C     8      C08   4000            synthemax_PFA
 7 SKNAS-EV  C     9      C09   8000            synthemax_PFA
 8 SKNAS-EV  C     10     C10   8000            synthemax_PFA
 9 SKNAS-EV  C     11     C11   12000           synthemax_PFA
10 SKNAS-EV  C     12     C12   12000           synthemax_PFA
# ℹ 212 more rows

$Assay_Plate11
# A tibble: 220 × 6
   cell_line row   column well  seeding_density condition    
   <chr>     <chr> <chr>  <chr> <chr>           <chr>        
 1 CHLA-186  C     3      C03   1000       

## Generate platemaps labelled with seeding density

In [4]:
for (plate in names(platemap_dfs)) {
    # Get the updated plate name
    updated_plate <- gsub("_Plate", " Plate ", plate)

    # Remove .png extension and add new suffixes
    filename_without_ext <- tools::file_path_sans_ext(output_platemap_files[[plate]])
    output_file_density <- paste0(filename_without_ext, "_seeding_density.png")
    
    platemap <- platetools::raw_map(
        data = as.numeric(platemap_dfs[[plate]]$seeding_density),
        well = platemap_dfs[[plate]]$well,
        plate = 384,
        size = 6
        ) +
        ggtitle(paste(updated_plate, "layout based on seeding density")) +
        theme(plot.title = element_text(size = 10, face = "bold")) +
        ggplot2::scale_fill_gradient2(
            name = "Seed Density",
            low = "white",
            high = "red",
        )  

    ggsave(
        output_file_density,
        platemap,
        dpi = 500,
        height = 3.5,
        width = 6
    )
}

Warning message:
“Warning: Your well label count ( 60 ) covers less than half the selected plate( 384 ).
Are you sure argument 'plate' is correct for the number of wells in your data?
note: Default is a 96-well plate.”


Warning message:
“Warning: Your well label count ( 180 ) covers less than half the selected plate( 384 ).
Are you sure argument 'plate' is correct for the number of wells in your data?
note: Default is a 96-well plate.”


Warning message:
“Warning: Your well label count ( 140 ) covers less than half the selected plate( 384 ).
Are you sure argument 'plate' is correct for the number of wells in your data?
note: Default is a 96-well plate.”


Warning message:
“Warning: Your well label count ( 100 ) covers less than half the selected plate( 384 ).
Are you sure argument 'plate' is correct for the number of wells in your data?
note: Default is a 96-well plate.”


Warning message:
“Warning: Your well label count ( 60 ) covers less than half the selected plate( 384 ).
Are you sure argument 'plate' is correct for the number of wells in your data?
note: Default is a 96-well plate.”


Warning message:
“Warning: Your well label count ( 180 ) covers less than half the selected plate( 384 ).
Are you sure argument 'plate' is correct for the number of wells in your data?
note: Default is a 96-well plate.”


In [5]:
for (plate in names(platemap_dfs)) {
    # Get the updated plate name
    updated_plate <- gsub("_Plate", " Plate ", plate)

    # Remove .png extension and add new suffixes
    filename_without_ext <- tools::file_path_sans_ext(output_platemap_files[[plate]])
    output_file_cell_line <- paste0(filename_without_ext, "_cell_line.png")
    
    # Update the plot to map by cell_line
    platemap <- platetools::raw_map(
        data = platemap_dfs[[plate]]$cell_line,  # Use cell line data directly
        well = platemap_dfs[[plate]]$well,
        plate = 384,
        size = 6
        ) +
        ggtitle(paste(updated_plate, "layout based on cell line")) +
        theme(plot.title = element_text(size = 10, face = "bold")) +
        ggplot2::geom_point(aes(fill = factor(platemap_dfs[[plate]]$cell_line)), shape = 21, size = 3) +  # Fill by cell line
        ggplot2::scale_fill_manual(values = colorRampPalette(RColorBrewer::brewer.pal(12, "Paired"))(length(unique(platemap_dfs[[plate]]$cell_line))),
                                   name = "Cell Line") +
        theme(
            legend.position = "right",             # Keep legend on the right side
            legend.justification = "center",       # Keep the legend centered
            legend.spacing.y = unit(0.4, "cm"),    # Adjust space between legend items
            legend.key.size = unit(0.6, "cm"),     # Adjust size of legend keys
            legend.box = "vertical",               # Vertical layout for the legend items
            legend.direction = "vertical"          # Ensures vertical flow
        ) +
        guides(fill = guide_legend(ncol = 2))      # Set legend to 2 columns

    # Save the updated plot with cell line coloring
    ggsave(
        output_file_cell_line,
        platemap,
        dpi = 500,
        height = 3.5,
        width = 6.5
    )
}


Warning message:
“Warning: Your well label count ( 60 ) covers less than half the selected plate( 384 ).
Are you sure argument 'plate' is correct for the number of wells in your data?
note: Default is a 96-well plate.”


Warning message:
“Warning: Your well label count ( 180 ) covers less than half the selected plate( 384 ).
Are you sure argument 'plate' is correct for the number of wells in your data?
note: Default is a 96-well plate.”


Warning message:
“Warning: Your well label count ( 140 ) covers less than half the selected plate( 384 ).
Are you sure argument 'plate' is correct for the number of wells in your data?
note: Default is a 96-well plate.”


Warning message:
“Warning: Your well label count ( 100 ) covers less than half the selected plate( 384 ).
Are you sure argument 'plate' is correct for the number of wells in your data?
note: Default is a 96-well plate.”


Warning message:
“Warning: Your well label count ( 60 ) covers less than half the selected plate( 384 ).
Are you sure argument 'plate' is correct for the number of wells in your data?
note: Default is a 96-well plate.”


Warning message:
“Warning: Your well label count ( 180 ) covers less than half the selected plate( 384 ).
Are you sure argument 'plate' is correct for the number of wells in your data?
note: Default is a 96-well plate.”
